# GP model mixing with `Taweret`: application to the dense matter equation of state

In this notebook, we will be investigating how to model mix the equations of state (EOSs) from chiral EFT and perturbative QCD for asymmetric, or neutron star, matter. In the field of dense matter physics, we have precise calculations from these EOSs at extreme ends of the density range the QCD phase diagram covers; however, in the intermediate region, we have very little models or data that have systematic uncertainties that we trust. We will use Bayesian model mixing to try to bridge this gap and learn something about what could possibly be the EOS result at these intermediate densities. This notebook is based on the work published in [this paper](https://arxiv.org/abs/2404.06323), [this paper](https://arxiv.org/abs/2505.18921), and [my thesis](https://etd.ohiolink.edu/acprod/odb_etd/r/etd/search/10?p10_accession_num=ohiou1752925796224309&clear=10&session=24884329572683).

Here we will start by using a stationary kernel to perform the model mixing, and in the next notebook we will change this out for a more sophisticated, non-stationary kernel. We will also investigate what happens when extra data is added to the training set, and how this affects the results of the mixing. We will use the BAND model mixing package `Taweret` to do this. Play around with the package and see what you can do!

In [ ]:
# import needed packages
import scipy as sp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.axes import Axes

# ! pip install sklearn
from sklearn.gaussian_process.kernels import RBF, Matern, RationalQuadratic, ConstantKernel as C, WhiteKernel
from sklearn.gaussian_process import GaussianProcessRegressor

# install Taweret and import it 
! pip install Taweret
from Taweret.core.base_model import BaseModel
from Taweret.mix.gp_bmm import GPmixing, GPPriors

In [ ]:
# important settings
n0 = 0.164   # fm^-3, saturation density

# color plotting assistance
orders = [0, 1, 2, 3]
order_labels = [r'$\chi$EFT (N$^{3}$LO)', r'pQCD (N$^{2}$LO)', 'Mixed Model']
color_list = ['Reds', 'Blues','Purples', 'Oranges', 'Greens', 'Greys']
cmaps = [plt.get_cmap(name) for name in color_list[:len(orders)]]
colors = [cmap(0.7 - 0.1 * (i == 0)) for i, cmap in enumerate(cmaps)]

Let's load all of the data sets we'll need first. This is going to pull from my GitHub public repository, where I have saved these files so you can access them. This notebook is also stored there.

In [ ]:
# access the data for this notebook from the GitHub repo URL
import urllib.request

files = [
    "chiral_data.npz",
    "pqcd_data.npz",
    "combined_data.npz",
    "bands_eos.npz",
    "combined_data_plus_heavy_ions.npz",
]

for filename in files:
    url = f"https://raw.githubusercontent.com/asemposki/band-camp-2026/main/tutorials/{filename}"
    urllib.request.urlretrieve(url, filename)

print("All data downloaded!")

Now that those files are imported into Colab, you can simply load them as usual---we do this next.

## EOS Data

This data are sampled points from the mean values of the chiral EFT results from [Drischler et al. (2021)](https://arxiv.org/abs/2009.06441) at N3LO and perturbative QCD results from [Gorda et al. (2023)](https://arxiv.org/pdf/2307.08734) at N2LO. They include the standard deviations and the covariances between the sampled points in the datasets. The data are in dictionaries with the keys 'dens', 'mean', 'std', and 'cov'. We will need these correlations to train the GP kernel parameters, since the GP will perform much better when told how the correlations between data points are structured. We note that here we are NOT including correlations between $\chi$EFT and pQCD; we assume these theories are uncorrelated, hence the full training data covariance matrix below is block-diagonal. 

In [ ]:
chiral_data = np.load('chiral_data.npz')
pqcd_data = np.load('pqcd_data.npz')
bands_data = np.load('bands_eos.npz')
combined_data = np.load('combined_data.npz')

# set covariance name now with nugget for stability
alpha_cov = combined_data['cov'] + 1e-8*np.diag(np.ones(len(combined_data['cov'])))

In [ ]:
# plot the data to see what it looks like
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(chiral_data['dens']/n0, chiral_data['mean'], chiral_data['std'], marker='o', linestyle=' ', color='red', label=r'$\chi$EFT training data')
plt.errorbar(pqcd_data['dens']/n0, pqcd_data['mean'], pqcd_data['std'], marker='o', linestyle=' ', color='blue', label=r'pQCD training data')
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xscale('log')
plt.legend()
plt.show()

Let's look at the combined data set, which is supposed to be the combination of all the points selected above. Let's check this.

In [ ]:
# plot the data to see what it looks like
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(chiral_data['dens']/n0, chiral_data['mean'], chiral_data['std'], marker='o', linestyle=' ', color='red', label=r'$\chi$EFT training data')
plt.errorbar(pqcd_data['dens']/n0, pqcd_data['mean'], pqcd_data['std'], marker='o', linestyle=' ', color='blue', label=r'pQCD training data')
plt.errorbar(combined_data['dens']/n0, combined_data['mean'], combined_data['std'], color='k', marker='+', linestyle=' ', label='Combined dataset')
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xscale('log')
plt.legend()
plt.show()

Now we want to set up `Taweret` to be able to handle the data we have. Let's call the class below and check what exists in the initialization.

## Setting up EOS models and Taweret class

In [ ]:
# load the docstrings
help(GPmixing)

We will need to set up the models and the input space first, before using `Taweret`.

In [ ]:
# make the prediction input set
density = np.linspace(1e-6, 16.0, 1000)

In [ ]:
# define the models here
class ChiralModel(BaseModel):

    def __init__(self, dens):

        self.dens = dens

        return None

    # usually a function but just data this time
    def chiral_model(self):

        self.chiral_mean = chiral_data['mean']
        self.chiral_std = chiral_data['std']
        self.chiral_cov = chiral_data['cov']
        return self.chiral_mean

    def evaluate(self):
        return self.chiral_mean

    def set_prior(self):
        '''
        Set the prior on model parameters.
        Not needed for this model.
        '''
        return None

class PQCDModel(BaseModel):

    def __init__(self, dens):

        self.dens = dens

        return None

    # usually a function but just data this time
    def pqcd_model(self):

        self.pqcd_mean = pqcd_data['mean']
        self.pqcd_std = pqcd_data['std']
        self.pqcd_cov = pqcd_data['cov']
        return self.pqcd_mean

    def evaluate(self):
        return self.pqcd_mean

    def set_prior(self):
        '''
        Set the prior on model parameters.
        Not needed for this model.
        '''
        return None

In [ ]:
# model dict from above
models = {
    '1': ChiralModel.chiral_model,
    '2': PQCDModel.pqcd_model,
}

Now the class will be able to see the models that we are using---though in this method we do not directly use them in this way, `Taweret` generally is constructed to use models organized into classes like this, so we do this here as an example.

After we set up the models, we now need to pick our Gaussian process kernel and appropriate priors. We do this below, recalling that a GP is described by a mean function and a kernel, or *covariance function*. We write our underlying theory we want to describe via a GP as

$$
f(x) \sim \mathcal{GP}[m(x), k(x,x')],
$$

where in our case we begin by picking $m(x) = 0$. We will, later in the tutorial, pick this to be a different function.

## GP kernels: the stationary RBF kernel and some priors

We are ready to set up the kernel we will be using for this model mixing method. We call below the squared-exponential radial basis function (RBF) kernel, which contains one free parameter to be fit during the training step, and which we multiply by another parameter controlling the amplitude of the GP. It is also infinitely differentiable, hence it has very smooth properties, which can sometimes cause issues, which we will see later in this tutorial. It is mathematically described as

$$
\bar{c}^{2} k(x, x'; \ell) = \bar{c}^{2} \exp{\left( \frac{-(x-x')^2}{2 \ell^2} \right)},
$$

where $\bar{c}$ is the marginal variance describing fluctuations from the mean of the GP in the data, and $\ell$ is the length scale of the GP, which parametrises the correlations between points in the data. Both of these hyperparameters will require priors to control their ranges. We will first try to fit the GP without them, show what happens, and then correct for this oversight.


A *stationary* kernel implies that this kernel is only dependent upon the distance between data points, as above. We will explore *non-stationary* kernels below in another section of this tutorial. For now, we will try to perform our model mixing using a stationary RBF kernel.

In [ ]:
# define the RBF kernel using scikit-learn in log scale (helps with training)
rbf_kernel = C(constant_value=1.1, constant_value_bounds=[0.1,2.25]) * RBF(length_scale=0.9, length_scale_bounds=[0.89, 3.0])

Like we mentioned above, we first take this kernel without any priors set other than the bounds above that constrain the parameters. Below, we will train on our training data from both chiral EFT and pQCD, including the covariances in the GP, and then we will plot our results and see how they look.

__Note__: We have set the values of the kernel parameters above for a log-scaled density input, because our problem is best solved when in log-space. This is due to the long distance between chiral EFT and pQCD when not in log-space, which is somewhat reduced when the log is taken. Here, we take the log of the density, train the kernel, and exponentiate the resulting predictions back to linear space. Hence, we give the GP mixing class the __log density__, not the linear density, since this kernel is in log density now, and we exponentiate afterward.

In [ ]:
# call Taweret and set up the class initialisation (selecting priors to be False here at first)
gpbmm = GPmixing(x=np.log(density), models=models, kernel=rbf_kernel, alpha=alpha_cov, 
                 priors=False)

In [ ]:
# let's look at the draws from the prior first to see what they are like
unconstrained_prior = gpbmm.prior_predict()

# plot these draws
# draw from the prior
_, draws = gpbmm.prior_predict(sample=True, n_samples=20)

#basic plot to check choices
fig = plt.figure(figsize=(8,6), dpi=100)
ax = plt.axes()
ax.tick_params(axis='x', direction='in')
ax.tick_params(axis='y', direction='in')
ax.locator_params(nbins=8)
ax.set_title('Prior draws')
ax.set_xlabel(r'n_B/n_0')
ax.set_ylabel('Draws')
ax.plot(density/n0, draws)
ax.set_xscale('log')
ax.set_xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.show()

We can see the smoothness of the kernel here from the draws we are making above. Now we train the kernel.

In [ ]:
# take the GPmixing class and train and predict using it 
gpbmm.train(np.log(combined_data['dens']).reshape(-1,1), combined_data['mean'])

In [ ]:
# predict with the mixing class above after looking at the trained parameters in log space
print(gpbmm.gpr.kernel_)

predictions = gpbmm.predict()

In [ ]:
# plot the results here to see how they look
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(chiral_data['dens']/n0, chiral_data['mean'], chiral_data['std'], marker='o', linestyle=' ', color='red', label=r'$\chi$EFT training data')
plt.errorbar(pqcd_data['dens']/n0, pqcd_data['mean'], pqcd_data['std'], marker='o', linestyle=' ', color='blue', label=r'pQCD training data')
plt.errorbar(combined_data['dens']/n0, combined_data['mean'], combined_data['std'], color='k', marker='+', linestyle=' ', label='Combined dataset')
plt.plot(np.exp(predictions['x'])/n0, predictions['mean'], color='mediumpurple', label=r'GP prediction')
plt.fill_between(np.exp(predictions['x'])/n0, predictions['mean'] - predictions['std'], predictions['mean'] + predictions['std'], color='mediumpurple', alpha=0.2)
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.ylim(0.0, 1.2)
plt.xscale('log')
plt.legend()
plt.show()

Clearly, since this mean result absolutely misses the data from pQCD, something is amiss! What do you think is happening?

__EXERCISE__: Talk to your neighbour about what could be going wrong. Do you both agree? How might this be happening and how could we fix it?

---

### Adding a prior

Now we are going to try to add a prior to mitigate the issues we are seeing above. This will work on controlling the length scale of the kernel so that we can get a more reasonable result that is physically believable. To do this, we will use our understanding of the problem at hand, namely that the two models __should not__ be correlated much at all. So, to bake this into our GP, we will control the kernel such that pQCD and chiral EFT cannot "talk" to one another so loudly.

In [ ]:
# add a prior and the constraints on it here
rbf_params = {
    'sigma': {'mu': 1.0, 'sig': 0.25},
    'lengthscale': {'mu': 0.85, 'sig': 0.15}
}

In [ ]:
# option for selecting a different kernel and priors to see difference
gpbmmprior = GPmixing(x=np.log(density), models=models, kernel=rbf_kernel, alpha=alpha_cov, priors=True, prior_params=rbf_params, prior_type='rbfnorm')

gpbmmprior.train(np.log(combined_data['dens']).reshape(-1,1), combined_data['mean'])

print(gpbmmprior.gpr.kernel_)

predictions_wprior = gpbmmprior.predict()

In [ ]:
# plot over the RBF result to see what's going on here
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(chiral_data['dens']/n0, chiral_data['mean'], chiral_data['std'], marker='o', linestyle=' ', color='red', label=r'$\chi$EFT training data')
plt.errorbar(pqcd_data['dens']/n0, pqcd_data['mean'], pqcd_data['std'], marker='o', linestyle=' ', color='blue', label=r'pQCD training data')
plt.errorbar(combined_data['dens']/n0, combined_data['mean'], combined_data['std'], color='k', marker='+', linestyle=' ', label='Combined dataset')
plt.plot(np.exp(predictions['x'])/n0, predictions['mean'], color='mediumpurple', label=r'GP prediction')
plt.fill_between(np.exp(predictions['x'])/n0, predictions['mean'] - predictions['std'], predictions['mean'] + predictions['std'], color='mediumpurple', alpha=0.2)


### EXERCISE: FILL IN YOUR NEW PREDICTIONS HERE TO COMPARE WITH NO PRIOR ###


plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.ylim(0.0, 1.2)
plt.xscale('log')
plt.legend()
plt.show()

This looks a lot more accurate, especially since we are now capturing the pQCD data pretty well. However, this does show that the training data can matter, and with a stationary kernel, it can be especially tricky to fit a good mixed model when adding too much training data can quickly make a GP singular, adding too little misses the fit, and capturing the shape of a structured curve is difficult unless one pays a lot of attention to where the points are located. 

__EXERCISE__: This one is optional---if you want to adjust the prior values above, go ahead and see if you can make the fit better!

We will try a Matern kernel next, which is much less smooth, constructed from Bessel functions, and only differentiable once if you pick its root ($\nu$, below) to be $3/2$.

In [ ]:
# Matern kernel set up here with priors
matern_kernel = C(constant_value=1.1, constant_value_bounds=[0.3,2.25]) * Matern(nu=1.5, length_scale=2.3, length_scale_bounds=[2.16, 3.3])

# priors
matern_params = {
    'sigma': {'mu': 1.0, 'sig': 0.25},
    'lengthscale': {'mu': 2.5, 'sig': 0.15},
}

In [ ]:
maternbmm = GPmixing(x=np.log(density), models=models, kernel=matern_kernel, alpha=alpha_cov, prior_params=matern_params, priors=True, prior_type='matern32')

# let's look at the draws from the prior first to see what they are like
unconstrained_prior = maternbmm.prior_predict()

# draw from the prior
_, draws = maternbmm.prior_predict(sample=True, n_samples=20)

#basic plot to check choices
fig = plt.figure(figsize=(8,6), dpi=100)
ax = plt.axes()
ax.tick_params(axis='x', direction='in')
ax.tick_params(axis='y', direction='in')
ax.locator_params(nbins=8)
ax.set_title('Prior draws: Matern kernel')
ax.set_xlabel(r'n_B/n_0')
ax.set_ylabel('Draws')
ax.plot(density/n0, draws)
ax.set_xscale('log')
ax.set_xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.show()

__EXERCISE__: Fill in the rest of the code below to generate the Matern kernel predictions, and plot the results in comparison to those above. How do they compare? Discuss with your neighbour.

In [ ]:
### FILL IN GP MIXING CODE HERE ###
maternbmm.train(..., ...)
print(maternbmm.gpr.kernel_)

matern_pred = ...

In [ ]:
# plot over the RBF result to see what's going on here
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(chiral_data['dens']/n0, chiral_data['mean'], chiral_data['std'], marker='o', linestyle=' ', color='red', label=r'$\chi$EFT training data')
plt.errorbar(pqcd_data['dens']/n0, pqcd_data['mean'], pqcd_data['std'], marker='o', linestyle=' ', color='blue', label=r'pQCD training data')
plt.errorbar(combined_data['dens']/n0, combined_data['mean'], combined_data['std'], color='k', marker='+', linestyle=' ', label='Combined dataset')
plt.plot(np.exp(predictions['x'])/n0, predictions['mean'], color='mediumpurple', label=r'GP prediction')
plt.fill_between(np.exp(predictions['x'])/n0, predictions['mean'] - predictions['std'], predictions['mean'] + predictions['std'], color='mediumpurple', alpha=0.2)


### FILL IN YOUR NEW PREDICTIONS HERE ###


plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.ylim(0.0, 1.2)
plt.xscale('log')
plt.legend()
plt.show()

---

This produces a much more accurate mean value and uncertainty estimation in the chiral EFT region above saturation density ($n_0 = 0.164$ fm$^{-3}$) and in the intermediate region. However, the pQCD region has some ballooning going on in between the data points, which we would rather avoid, and which indicates the length scale is a little too short in that region.

However, there is a bigger problem: the region at the very low end of chiral EFT is poorly captured here, and will be poorly captured by any stationary kernel. This happens because the stationary kernel is unable to capture the changing length scales of our problem across the full density domain. If you have been paying close attention, you may have guessed this would be a problem from the beginning---our density input space is quite large, and expecting one single kernel to span the full space is a bit too hopeful. Chiral EFT does not have the same long, logarithmic correlation length that pQCD has, so we will need to try a non-stationary kernel to capture both of these effects simultaneously. Below is a walkthrough of one type of these kernels, a *changepoint* kernel, that is incorporated into `Taweret`. 

---
---

# Using a non-stationary kernel: the dense matter EOS revisited

Now we will design what we will call a nonstationary kernel, which means it will depend on the input space (here, on our location in ). To do this, we first will define frozen GP kernels that will emulate our models. We do this here because it allows us to quickly design a kernel that has the necessary positive, semi-definite characteristic the GP relies on. We do this below by first defining our frozen kernels and plotting their results on top of our models. These frozen kernels are RBF kernels equipped with a ConstantKernel to absorb the marginal variance.

Now it's your turn to do more of the work! I will set up some of the code for you, but this time I want you to walk through the exercises a bit more independently. 

In [ ]:
# import the GP wrapper that we need for this part directly
from Taweret.mix.gp_bmm import GPRwrapper

In [ ]:
# frozen kernel for chiral EFT
chi_c = 0.9
chi_ls = 0.31
k_chi = (C(constant_value=chi_c, constant_value_bounds='fixed') \
         * RBF(length_scale=chi_ls, length_scale_bounds='fixed'))

gpchi = GPRwrapper(
    kernel=k_chi,
    alpha=chiral_data['cov'],
)

# note here we are not using log space anymore!!
gpchi.fit(chiral_data['dens'].reshape(-1,1), chiral_data['mean'].reshape(-1,1))

print(gpchi.kernel_)

chi_predict, chi_std_predict = gpchi.predict(density.reshape(-1,1), return_std=True)
_, chi_cov_predict = gpchi.predict(density.reshape(-1,1), return_cov=True)

# chiral band
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red', label=r'ChEFT (N3LO)')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], 
                 bands_data['meanchiral']+bands_data['stdchiral'],
                alpha=0.2, color='r')

# data points used for fit
plt.errorbar(x=chiral_data['dens']/n0, y=chiral_data['mean'], yerr=chiral_data['std'], linestyle='none', marker='o', color='k', 
             label='Training points')

# our results
plt.plot(density/n0, chi_predict, color='mediumpurple', zorder=10, label=r'GP fit')
plt.fill_between(density/n0, chi_predict-chi_std_predict, chi_predict+chi_std_predict,
                 color='mediumpurple', alpha=0.2)
plt.ylim(0.0, 0.4)
plt.xlim(min(bands_data['denschiral'])/n0, max(bands_data['denschiral'])/n0)
plt.xscale('log')
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.legend(loc='upper left')
plt.show()

This misses the data a bit at the lower end, but we also are planning on attaching the crust around that region when we run our neutron star calculations, so we do not have to worry too much about missing that region with the GP fit.

In [ ]:
# frozen kernel for pQCD
pqcd_c = 1.0
pqcd_ls = 5.0
k_pqcd = (C(constant_value=pqcd_c, constant_value_bounds='fixed') \
          * RBF(length_scale=pqcd_ls, length_scale_bounds='fixed'))

# pQCD
gppqcd = GPRwrapper(
    kernel=k_pqcd,
    alpha = pqcd_data['cov'],
)

gppqcd.fit(pqcd_data['dens'].reshape(-1,1), pqcd_data['mean'].reshape(-1,1))

pqcd_predict, pqcd_std_predict = gppqcd.predict(density.reshape(-1,1), return_std=True)
_, pqcd_cov_predict = gppqcd.predict(density.reshape(-1,1), return_cov=True)

# plot the result quickly to test
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue', label=r'pQCD (N2LO)')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], 
                 bands_data['meanpqcd']+bands_data['stdpqcd'], alpha=0.2, color='blue')
plt.errorbar(x=pqcd_data['dens']/n0, y=pqcd_data['mean'], yerr=pqcd_data['std'], linestyle='none', marker='o', color='k',
             label=r'Training data')
plt.plot(density/n0, pqcd_predict, color='mediumpurple', zorder=10, label='GP fit')
plt.fill_between(density/n0, pqcd_predict-pqcd_std_predict, pqcd_predict+pqcd_std_predict, \
                 color='mediumpurple', alpha=0.2)
plt.ylim(1.0, 1.2)
plt.xlim(10.0, 100.0)
plt.legend(loc='upper right')
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.show()

Let's now plot the two kernels in the same plot with the GPs extended over the full density range.

In [ ]:
# check the plotting of both of these kernels together
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red', label=r'ChEFT (N3LO)')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], 
                 bands_data['meanchiral']+bands_data['stdchiral'],
                alpha=0.2, color='red')
plt.errorbar(x=combined_data['dens']/n0, y=combined_data['mean'], yerr=combined_data['std'], marker='o', color='k', linestyle='none')
plt.plot(density/n0, chi_predict, color='mediumpurple', zorder=10)
plt.fill_between(density/n0, chi_predict-chi_std_predict, chi_predict+chi_std_predict, \
                 color='mediumpurple', alpha=0.2)
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue', label=r'pQCD (N2LO)')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], 
                 alpha=0.2, color='blue')
plt.errorbar(x=combined_data['dens']/n0, y=combined_data['mean'], yerr=combined_data['std'], marker='o', color='k', linestyle='none')
plt.plot(density/n0, pqcd_predict, color='mediumpurple', zorder=10)
plt.fill_between(density/n0, pqcd_predict-pqcd_std_predict, pqcd_predict+pqcd_std_predict, color='mediumpurple', alpha=0.2)
plt.ylim(0.0, 1.4)
plt.xlim(0.31, 100.0)
plt.xscale('log')
plt.legend()
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'$P(n_B)/P_{FG}(n_B)$')
plt.show()

Now that we've established that these frozen kernels do an adequate job, we will use them in the changepoint kernel below. Now I want you to set up the training and priors and see what you get!

## BMM step: changepoint kernel with a sigmoid mixing function

We define the changepoint kernel at locations $x_i$, $x_j$, including the two frozen kernels, $k_1$ and $k_2$, which serve as the two anchors at the edges of our GP training region:

$$
k(x_i, x_j) = (1 - \sigma(x_i)) k_{1}(x_i,x_j) (1 - \sigma(x_j)) + \sigma(x_i) k_{2}(x_i, x_j) \sigma(x_j),
$$

where, $\sigma$ here is a sigmoid mixing function, given as

$$
\sigma(x) = \frac{1}{1 + \exp(-(x-x_{0})/w)}.
$$

The changepoint $x_0$ in the sigmoid function controls the location of the crossover between $k_1$ and $k_2$, and the width parameter $w$ describes the slope of the function about the changepoint location. These are two hyperparameters that will need to be optimized in the full training of the kernel. Because this kernel is non-stationary, it is able to do this changeover from chiral EFT to pQCD and is therefore parametrized with respect to the density instead of the Euclidean distance---this means that we can directly interpret our parameters in terms of location in density unlike the stationary kernel. 

In [ ]:
# load up the non-stationary kernel options and discuss
from Taweret.utils.kernels import SigmoidChangepoint

help(SigmoidChangepoint)

Now we set up this kernel and the priors below.

In [ ]:
# kernel set up and priors
kernelCP = SigmoidChangepoint(
    ls1=chi_ls, ls2=pqcd_ls, cbar1=chi_c, cbar2=pqcd_c,
    changepoint=0.9, changepoint_bounds=[0.488, 3.12], width=0.16, width_bounds=[0.05, 0.2]
)

# set up the prior params we will use (guesses for our scenario)
cp_params = {
    'cp': {'mu': 0.98, 'sig': 0.33},
    'w': {'mu': 0.16, 'sig': 0.155},
}

# set up prior dict for the different hyperparameters
prior_dict = {
    'w': 'truncnorm',
    'cp': 'truncnorm',
    'switch': 'sigmoid',
}

We will not take `prior_params=None`, but we will use the `cp_params` defined above, since we set `prior_choice='changepoint'`. A user can of course enter their own priors for the GP to use! The current hyperprior form being used is a truncated normal distribution,

$$
x_0 | I \sim \mathcal{U}(a, b) \times \mathcal{N}(\mu, \sigma),
$$
and the same form for the width $w$. Here, $a$ and $b$ can be implemented as hard cutoffs based on physical restrictions from the problem; in our case, we simply would pick something reasonable for the model range we are in so that we do not let the changepoint or width drift too far into the model spaces themselves. $\mu$ and $\sigma$ allow us to also control the general range of where we would expect the changepoint or width MAP value to be. This can also be based on physical restrictions; in this case, for example, one would expect the changepoint to be somewhere between where $\chi$EFT and pQCD fail, so the value $n_B=0.9$ is a reasonable guess given this information.

__EXERCISE__: Fill in the missing code below to train this GP, predict the model-mixed EOS, and plot the results!

In [ ]:
# set up the model mixing step   ### FILL IN THE MISSING INFORMATION ###
gpchangepoint = GPmixing(
    x=density, models=models, alpha=combined_data['cov'], kernel=...,
    priors=True, prior_params=..., prior_choice='changepoint', prior_type=prior_dict,
    switch='sigmoid', nopt=10000)

In [ ]:
# train the GP
### FILL IN YOUR CODE HERE ###

print(gpchangepoint.gpr.kernel_) # shows fitted kernel results

# predict using the fitted GP
### FILL IN YOUR CODE HERE ###

In [ ]:
# plot against the original stationary results
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], 
                 bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], 
                 bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(combined_data['dens']/n0, combined_data['mean'], combined_data['std'], color='k', marker='+', 
             linestyle=' ', label='Combined dataset')


### ADD YOUR RESULT HERE ###


plt.plot(density/n0, chi_predict, color='mediumpurple', zorder=10)
plt.fill_between(density/n0, chi_predict-chi_std_predict, chi_predict+chi_std_predict, color='mediumpurple', alpha=0.2)
plt.plot(density/n0, pqcd_predict, color='mediumpurple', zorder=10)
plt.fill_between(density/n0, pqcd_predict-pqcd_std_predict, pqcd_predict+pqcd_std_predict, color='mediumpurple', alpha=0.2)

plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.ylim(0.0, 1.2)
plt.xscale('log')
plt.legend()
plt.show()

This looks a little funky, especially since the mean function is zero---why do you think this matters for the result we get above? 

__EXERCISE__: Discuss with your neighbour why the mean function being zero here affects the model-mixed EOS in the intermediate region so much.

However, this is a good starting point for using a changepoint kernel. This shows clearly that the crossover between the two frozen kernels is right in the intermediate region, and it also ensures we obtain the correct results in the two extreme regions where have EOSs to model them. Since we are inputting so little information about the intermediate region into our model mixing here, it makes sense that we don't learn a whole lot about it other than that the uncertainties are quite large. However, what if we had some data to use? How would that affect our result? Let's see below!

---
---

## Using heavy-ion data: how does this change the prediction?

Now we want to add some data to the problem to see how the prediction changes. We will use a draw from the contour in [Huth et al.](https://arxiv.org/abs/2107.06229) that employed heavy-ion collision data in their study. We'll start by loading this data, including our combined dataset from previously.

In [ ]:
# load the data with the heavy-ion collisions included
combined_data_hic = np.load('combined_data_plus_heavy_ions.npz')

In [ ]:
# plot the data to see what it looks like
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red', label=r'$\chi$EFT (N$^{3}$LO)')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue', label='pQCD (N$^{2}$LO)')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(combined_data_hic['dens']/n0, combined_data_hic['mean'], combined_data_hic['std'], color='k', marker='o', linestyle='none', label=r'Combined data')
plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xscale('log')
plt.legend()
plt.show()

Now I want you to continue the process as previously: initialize the GPmixing class with the same models and settings as for the changepoint kernel we already did above, except now we include the heavy-ion data point in the (a) covariance matrix going into `alpha`, and (b) the GP fit on the data.

In [ ]:
### ADD YOUR CODE HERE ###


# plot results over top of old result without heavy-ion information
# plot against the original stationary results
fig = plt.figure(figsize=(6,5), dpi=150)
plt.plot(bands_data['denschiral']/n0, bands_data['meanchiral'], color='red')
plt.plot(bands_data['denspqcd']/n0, bands_data['meanpqcd'], color='blue')
plt.fill_between(bands_data['denschiral']/n0, bands_data['meanchiral']-bands_data['stdchiral'], 
                 bands_data['meanchiral']+bands_data['stdchiral'], alpha=0.2, color='red')
plt.fill_between(bands_data['denspqcd']/n0, bands_data['meanpqcd']-bands_data['stdpqcd'], 
                 bands_data['meanpqcd']+bands_data['stdpqcd'], color='blue', alpha=0.2)
plt.errorbar(combined_data_hic['dens']/n0, combined_data_hic['mean'], combined_data_hic['std'], color='k', marker='+', 
             linestyle=' ', label='Combined dataset')



### ADD YOUR RESULT HERE ###



plt.plot(predict_cp['x']/n0, predict_cp['mean'], color='green', label='GP prediction')
plt.fill_between(predict_cp['x']/n0, predict_cp['mean']-predict_cp['std'], predict_cp['mean']+predict_cp['std'], color='green', alpha=0.2)

plt.plot(density/n0, chi_predict, color='mediumpurple', zorder=10)
plt.fill_between(density/n0, chi_predict-chi_std_predict, chi_predict+chi_std_predict, color='mediumpurple', alpha=0.2)
plt.plot(density/n0, pqcd_predict, color='mediumpurple', zorder=10)
plt.fill_between(density/n0, pqcd_predict-pqcd_std_predict, pqcd_predict+pqcd_std_predict, color='mediumpurple', alpha=0.2)

plt.xlabel(r'Scaled density $n_B/n_0$')
plt.ylabel(r'Scaled pressure $P(n_B)/P_{FG}(n_B)$')
plt.xlim(min(chiral_data['dens'])/n0, max(pqcd_data['dens'])/n0)
plt.ylim(0.0, 1.2)
plt.xscale('log')
plt.legend()
plt.show()

__EXERCISE__: How does including the heavy-ion data point change the model-mixed EOS result? What can you infer from this if we were in the situation where we had even more data?

---
---

# Take-home exercise: employing a non-zero mean function

In this tutorial, we've covered a lot of GP mixing techniques and how to adapt them to the current physics problem we're interested in, the nuclear matter EOS. We can do a lot more with this, including adding a non-zero mean function to help us characterize the intermediate region between $\chi$EFT and pQCD. If you are ambitious, you can go ahead and try implementing one, and see if you get something like we obtained in our published work that this tutorial is based on, found [here](https://arxiv.org/abs/2505.18921). You could even plug all this into a Tolman-Oppenheimer-Volkoff solver to obtain the mass-radius relationship for neutron stars, if you wanted to!

I hope this was a useful tutorial for you to get the hang of how to perform Bayesian model mixing with Gaussian processes! Good luck, and may the physics gods smile upon your research!

In [ ]:
### ADD YOUR CODE HERE ###